# RT-DETR KD — Teacher vs Student Attention Map Visualization

CS229 Project — Umut Onur Yasar

Visualizes decoder cross-attention maps for a set of COCO val images,
comparing teacher (RT-DETR-L / ResNet-50) and student (RT-DETR-S / ResNet-18)
side-by-side before and after knowledge distillation.


In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

import yaml
from src.models.rtdetr import build_rtdetr
from src.data.transforms import build_transforms

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ----- Configuration — update these paths -----
STUDENT_CFG      = '../configs/rtdetr_r18vd_coco.yml'
TEACHER_CFG      = '../configs/rtdetr_r50vd_coco.yml'
STUDENT_WEIGHTS  = '../runs/run08_feature_l1.0/checkpoint_best.pth'   # KD student
STUDENT_BASELINE = '../runs/run00_baseline/checkpoint_best.pth'        # Baseline student
TEACHER_WEIGHTS  = None  # Set path if you have teacher weights
VAL_IMG_DIR      = '/data/coco/val2017'
IMG_SIZE         = 640

# Sample image IDs from COCO val (pick ones with clear objects)
SAMPLE_IMAGE_IDS = [139, 785, 1000, 2153, 5001]  # adjust as needed

In [ ]:
def load_model(cfg_path: str, weights_path: str | None = None) -> torch.nn.Module:
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    model = build_rtdetr(cfg).to(DEVICE)
    if weights_path and Path(weights_path).exists():
        ckpt = torch.load(weights_path, map_location='cpu')
        state = ckpt.get('model_state_dict', ckpt)
        model.load_state_dict(state, strict=False)
        print(f'Loaded weights: {weights_path}')
    else:
        print(f'No weights loaded for {cfg_path}')
    model.eval()
    return model


# Load models
print('Loading teacher...')
teacher = load_model(TEACHER_CFG, TEACHER_WEIGHTS)

print('Loading KD student...')
student_kd = load_model(STUDENT_CFG, STUDENT_WEIGHTS)

print('Loading baseline student...')
student_base = load_model(STUDENT_CFG, STUDENT_BASELINE)

print('All models loaded.')

In [ ]:
val_transforms = build_transforms(train=False, img_size=IMG_SIZE)

def preprocess_image(img_path: str | Path) -> tuple[torch.Tensor, Image.Image]:
    """Load and preprocess a single image."""
    img = Image.open(img_path).convert('RGB')
    # Dummy target for transforms
    dummy_target = {
        'boxes': torch.zeros((0, 4)),
        'labels': torch.zeros((0,), dtype=torch.long),
    }
    tensor, _ = val_transforms(img, dummy_target)
    return tensor.unsqueeze(0).to(DEVICE), img


@torch.no_grad()
def get_attention_maps(
    model: torch.nn.Module,
    img_tensor: torch.Tensor,
    top_k_queries: int = 10,
    layer_idx: int = -1,
) -> tuple[np.ndarray, np.ndarray]:
    """Run model and extract cross-attention maps for top-k confident queries.

    Returns:
        (scores, attn_maps): each of shape [top_k] and [top_k, H_feat, W_feat]
    """
    outputs = model(img_tensor)
    attn_maps = model.get_attn_maps_tensor()  # [L, 1, nhead, Q, N]

    # Select layer
    L = attn_maps.size(0)
    layer = layer_idx if layer_idx >= 0 else L + layer_idx
    layer = max(0, min(layer, L - 1))
    attn = attn_maps[layer, 0]  # [nhead, Q, N]
    attn = attn.mean(dim=0)     # [Q, N] — average over heads

    # Get top-k confident queries (by max class score)
    scores = outputs['pred_logits'][0].sigmoid().max(dim=-1).values  # [Q]
    top_k = min(top_k_queries, len(scores))
    top_scores, top_idx = scores.topk(top_k)

    selected_attn = attn[top_idx].cpu().numpy()  # [k, N]
    top_scores = top_scores.cpu().numpy()

    # Figure out spatial dimensions (heuristic: sqrt of N)
    # For 640x640 input: backbone gives 80x80 + 40x40 + 20x20 = 8400 tokens
    # We just reshape the dominant scale for visualization
    N = selected_attn.shape[-1]
    # Use the largest scale (first ~HW tokens)
    h = w = int(np.sqrt(N // 3 * 4)) if N > 0 else 8  # rough
    # Better: assume 80x80 portion dominates
    h_s = int(IMG_SIZE / 8)  # stride-8 scale
    w_s = h_s
    N_s = h_s * w_s  # tokens from C3
    if N >= N_s:
        spatial_attn = selected_attn[:, :N_s].reshape(top_k, h_s, w_s)
    else:
        side = int(np.sqrt(N)) if N > 0 else 1
        spatial_attn = selected_attn[:, :side*side].reshape(top_k, side, side)

    return top_scores, spatial_attn

In [ ]:
def visualize_attention_comparison(
    img: Image.Image,
    attn_teacher: np.ndarray,
    attn_base: np.ndarray,
    attn_kd: np.ndarray,
    scores_teacher: np.ndarray,
    scores_base: np.ndarray,
    scores_kd: np.ndarray,
    img_id: int,
    top_k: int = 5,
) -> None:
    """Plot attention maps for top-k queries side by side."""
    # Show only top_k queries (already sorted)
    top_k = min(top_k, attn_teacher.shape[0], attn_base.shape[0], attn_kd.shape[0])

    fig, axes = plt.subplots(top_k, 4, figsize=(16, 3 * top_k))
    if top_k == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['Original Image', 'Teacher (ResNet-50)', 'Student Baseline', 'Student + Feature-KD']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=11, fontweight='bold')

    img_np = np.array(img)

    for row in range(top_k):
        # Original image
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_ylabel(f'Query {row+1}', fontsize=9)
        axes[row, 0].axis('off')

        for col, (attn_map, score, model_name) in enumerate([
            (attn_teacher[row], scores_teacher[row], 'Teacher'),
            (attn_base[row],    scores_base[row],    'Baseline'),
            (attn_kd[row],      scores_kd[row],      'KD Student'),
        ], start=1):
            # Upsample attention map to image size
            h_map, w_map = attn_map.shape
            attn_up = Image.fromarray(
                (attn_map / (attn_map.max() + 1e-8) * 255).astype(np.uint8)
            ).resize((img.width, img.height), Image.BILINEAR)
            attn_up = np.array(attn_up)

            # Overlay on image
            axes[row, col].imshow(img_np)
            axes[row, col].imshow(attn_up, cmap='jet', alpha=0.5, vmin=0, vmax=255)
            axes[row, col].set_title(f'score={score:.3f}', fontsize=8)
            axes[row, col].axis('off')

    plt.suptitle(f'COCO Image ID: {img_id}', fontsize=13, y=1.01)
    plt.tight_layout()
    save_path = Path(f'../attention_img{img_id}.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.show()
    print(f'Saved: {save_path}')

In [ ]:
# ----- Process each sample image -----
val_img_dir = Path(VAL_IMG_DIR)

for img_id in SAMPLE_IMAGE_IDS:
    # Find image file (COCO val filenames are zero-padded 12-digit)
    img_path = val_img_dir / f'{img_id:012d}.jpg'
    if not img_path.exists():
        print(f'Image {img_id} not found at {img_path}, skipping.')
        continue

    print(f'\nProcessing image {img_id}...')
    img_tensor, pil_img = preprocess_image(img_path)

    scores_teacher, attn_teacher = get_attention_maps(teacher, img_tensor)
    scores_base,    attn_base    = get_attention_maps(student_base, img_tensor)
    scores_kd,      attn_kd      = get_attention_maps(student_kd, img_tensor)

    # Align top-k across models to show same number of queries
    top_k = min(5, len(scores_teacher), len(scores_base), len(scores_kd))

    visualize_attention_comparison(
        pil_img,
        attn_teacher[:top_k], attn_base[:top_k], attn_kd[:top_k],
        scores_teacher[:top_k], scores_base[:top_k], scores_kd[:top_k],
        img_id=img_id, top_k=top_k,
    )

In [ ]:
# ----- Aggregate attention entropy analysis -----
# Lower entropy = more focused attention = better localization

def attention_entropy(attn_map: np.ndarray) -> float:
    """Compute normalized entropy of a 2D attention map."""
    flat = attn_map.flatten()
    p = flat / (flat.sum() + 1e-10)
    p = p[p > 0]
    entropy = -(p * np.log(p)).sum()
    max_entropy = np.log(len(flat))
    return float(entropy / max_entropy)


results = {'Teacher': [], 'Baseline': [], 'KD Student': []}

img_files = sorted(val_img_dir.glob('*.jpg'))[:20]  # sample 20 images
for img_path in img_files:
    try:
        img_tensor, _ = preprocess_image(img_path)
        for model, name in [(teacher, 'Teacher'), (student_base, 'Baseline'), (student_kd, 'KD Student')]:
            _, attn = get_attention_maps(model, img_tensor, top_k_queries=5)
            entropies = [attention_entropy(a) for a in attn]
            results[name].extend(entropies)
    except Exception as e:
        print(f'  Error on {img_path.name}: {e}')

print('=== Attention Entropy Analysis (lower = more focused) ===')
print(f'{"Model":<15} {"Mean Entropy":<15} {"Std":<10} {"N"}')
for name, vals in results.items():
    if vals:
        print(f'{name:<15} {np.mean(vals):<15.4f} {np.std(vals):<10.4f} {len(vals)}')

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
names = [k for k, v in results.items() if v]
means = [np.mean(results[k]) for k in names]
stds  = [np.std(results[k])  for k in names]
colors_bar = ['#FF9800', '#888888', '#E91E63']
bars = ax.bar(names, means, yerr=stds, capsize=5,
              color=colors_bar[:len(names)], width=0.5)
ax.set_ylabel('Normalized Attention Entropy')
ax.set_title('Attention Entropy: Teacher vs Student Comparison\n(lower = more focused)')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../attention_entropy.pdf', bbox_inches='tight')
plt.savefig('../attention_entropy.png', bbox_inches='tight', dpi=150)
plt.show()